In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/neurosense_cleaned.csv")

print(df.shape)
df.head()

In [ ]:

X = df.drop(columns=["label", "subject", "session", "trial", "sample"], errors="ignore")
y = df["label"]

X = X.select_dtypes(include=["number"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts())

groups = df["subject"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# Decision Tree bazik

dt_basic = DecisionTreeClassifier(
    random_state=42
)

dt_basic.fit(X_train, y_train)

y_pred_dt_basic = dt_basic.predict(X_test)

In [ ]:
# Vlerësimi i Decision Tree bazik

dt_basic_results = {
    "Model": "Decision Tree Basic",
    "Accuracy": accuracy_score(y_test, y_pred_dt_basic),
    "Precision": precision_score(y_test, y_pred_dt_basic, average="weighted"),
    "Recall": recall_score(y_test, y_pred_dt_basic, average="weighted"),
    "F1-score": f1_score(y_test, y_pred_dt_basic, average="weighted")
}

print("Decision Tree Basic Results")
for key, value in dt_basic_results.items():
    print(key, ":", value)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt_basic))

In [ ]:
#Confusion Matrix për Decision Tree bazik

cm_basic = confusion_matrix(y_test, y_pred_dt_basic)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_basic, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Decision Tree Basic")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
# Feature Importance

feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": dt_basic.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance_df.head(20)


In [ ]:
# Vizualizimi i Top 20 Feature Importance

top_20_features = feature_importance_df.head(20)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_20_features,
    x="Importance",
    y="Feature"
)

plt.title("Top 20 Feature Importances - Decision Tree")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

In [ ]:
# Hyperparameter Tuning për Decision Tree

param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_search_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=2
)

grid_search_dt.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search_dt.best_params_)

print("Best CV Score:")
print(grid_search_dt.best_score_)

In [ ]:
#  Vlerësimi final i Decision Tree pas tuning

best_dt_model = grid_search_dt.best_estimator_

y_pred_best_dt = best_dt_model.predict(X_test)

dt_final_results = {
    "Model": "Decision Tree + GridSearchCV",
    "Accuracy": accuracy_score(y_test, y_pred_best_dt),
    "Precision": precision_score(y_test, y_pred_best_dt, average="weighted"),
    "Recall": recall_score(y_test, y_pred_best_dt, average="weighted"),
    "F1-score": f1_score(y_test, y_pred_best_dt, average="weighted"),
    "Best Parameters": grid_search_dt.best_params_
}

print("Final Decision Tree Results")
for key, value in dt_final_results.items():
    print(key, ":", value)

print("\nFinal Classification Report:")
print(classification_report(y_test, y_pred_best_dt))

In [ ]:
#  Confusion Matrix final

cm_final = confusion_matrix(y_test, y_pred_best_dt)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_final, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Tuned Decision Tree")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
# 13. Krahasimi Basic vs Tuned

dt_comparison_df = pd.DataFrame([
    dt_basic_results,
    dt_final_results
])

dt_comparison_df


dt_results_df = pd.DataFrame([dt_final_results])

dt_results_df.to_csv("../results/tables/decision_tree_results.csv", index=False)
dt_comparison_df.to_csv("../results/tables/decision_tree_comparison_results.csv", index=False)

dt_results_df